# Segmentation analysis

version 3

## IMPORT

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import psycopg2

print(f"NumPy:     {np.__version__}")
print(f"Pandas:    {pd.__version__}")

# DB connections
load_dotenv()
local_engine  = create_engine(os.getenv("LOCAL_traveltide_URL"))
connection    = local_engine.connect().execution_options(
                    isolation_level="AUTOCOMMIT")
local_path = os.getenv("PROJECT_ROOT")
DATA_PATH = os.environ.get('DATA_PATH', 'data')
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline
sns.style='whitegrid'

Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
NumPy:     2.4.4
Pandas:    3.0.2


### DataFrame build

In [ ]:
# Load clustered user features
df_users = pd.read_sql(
    sa.text("SELECT * FROM tt_project.user_features"),
    local_engine)
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline
sns.style='whitegrid'

print(f"df_users shape: {df_users.shape}")
print(f"df_users_transformed shape: {df_users2.shape}")

## Determine SQL/Standard Perk rules

### Customer Lifecycle - Perk Assignment Strategy


Perk assignment uses a lifecycle-gated model based on `life_total_trips`.
Users with fewer than 3 completed trips receive simplified assignments
appropriate to their behavioral stage.  Users with 3+ trips receive full
behavioral evaluation across all five perk dimensions.

**Statistical basis:** Once is an event, twice is two events, three is a trend.  
Single-trip behavioral signals (cancel rates, international %) are statistically
unreliable at n=1 observation and excluded from lower-tier logic.

**Priority logic within Tier 3+:** Business priority order - revenue risk first
(No Change Fees), then retention value (Free Checked Bags), then behavioral
specificity (Free Hotel Meals, Exclusive Discounts), then loyalty recognition
(Premium Travel Benefits as earned residual).

Full documentation: TravelTide_Detailed_Report.md - Section 10

#### Tier 0 - Never Booked

In [13]:
q_tier0 = """
SELECT
    user_id,
    'Exclusive Discounts' AS assigned_perk,
    0 AS life_total_trips,
    'tier_0_never_booked' AS assignment_reason
FROM tt_project.user_features
WHERE life_total_trips = 0;
"""
df_tier0 = pd.read_sql(sa.text(q_tier0), connection)
display(df_tier0)
print(f"Tier 0 assigned: {len(df_tier0)} users")

,user_id,assigned_perk,life_total_trips,assignment_reason
0,167852,Exclusive Discounts,0,tier_0_never_booked
1,171470,Exclusive Discounts,0,tier_0_never_booked
2,182191,Exclusive Discounts,0,tier_0_never_booked
3,216796,Exclusive Discounts,0,tier_0_never_booked
4,217114,Exclusive Discounts,0,tier_0_never_booked
...,...,...,...,...
517,743151,Exclusive Discounts,0,tier_0_never_booked
518,746934,Exclusive Discounts,0,tier_0_never_booked
519,748759,Exclusive Discounts,0,tier_0_never_booked
520,770252,Exclusive Discounts,0,tier_0_never_booked


Tier 0 assigned: 522 users


#### Tier 1 - Single trip

In [14]:
q_tier1 = """
SELECT
    user_id,
    CASE 
        WHEN cancel_spend_ratio >= 0.40 THEN 'No Change Fees'
        ELSE 'Exclusive Discounts'
    END AS assigned_perk,
    1 AS life_total_trips,
    CASE 
        WHEN cancel_spend_ratio >= 0.40 THEN 'tier_1_single_trip_high_cancel'
        ELSE 'tier_1_single_trip_default'
    END AS assignment_reason
FROM tt_project.user_features
WHERE life_total_trips = 1
AND user_id NOT IN (
    SELECT user_id FROM tt_project.user_features
    WHERE life_total_trips = 0
);
"""
df_tier1 = pd.read_sql(sa.text(q_tier1), connection)
display(df_tier1)
print(f"Tier 1 assigned: {len(df_tier1)} users")
print(df_tier1['assigned_perk'].value_counts())
print(df_tier1['assignment_reason'].value_counts())

,user_id,assigned_perk,life_total_trips,assignment_reason
0,152583,Exclusive Discounts,1,tier_1_single_trip_default
1,160754,Exclusive Discounts,1,tier_1_single_trip_default
2,164522,Exclusive Discounts,1,tier_1_single_trip_default
3,229108,No Change Fees,1,tier_1_single_trip_high_cancel
4,258451,No Change Fees,1,tier_1_single_trip_high_cancel
...,...,...,...,...
934,752933,Exclusive Discounts,1,tier_1_single_trip_default
935,763129,Exclusive Discounts,1,tier_1_single_trip_default
936,763792,Exclusive Discounts,1,tier_1_single_trip_default
937,777366,Exclusive Discounts,1,tier_1_single_trip_default


Tier 1 assigned: 939 users
assigned_perk
Exclusive Discounts    839
No Change Fees         100
Name: count, dtype: int64
assignment_reason
tier_1_single_trip_default        839
tier_1_single_trip_high_cancel    100
Name: count, dtype: int64


#### Tier 2 - Two trips

In [17]:
q_tier2 = """
SELECT
    user_id,
    CASE
        WHEN COALESCE(cancel_spend_ratio, 0) >= 0.40 THEN 'No Change Fees'
        WHEN COALESCE(pct_complete_flights_intl, 0) = 1.0 THEN 'Free Checked Bags'
        ELSE 'Exclusive Discounts'
    END AS assigned_perk,
    2 AS life_total_trips,
    CASE
        WHEN COALESCE(cancel_spend_ratio, 0) >= 0.40 THEN 'tier_2_two_trip_high_cancel'
        WHEN COALESCE(pct_complete_flights_intl, 0) = 1.0 THEN 'tier_2_two_trip_exclusive_intl'
        ELSE 'tier_2_two_trip_default'
    END AS assignment_reason
FROM tt_project.user_features
WHERE life_total_trips = 2
AND user_id NOT IN (
    SELECT user_id FROM tt_project.user_features
    WHERE life_total_trips <= 1
);
"""
df_tier2 = pd.read_sql(sa.text(q_tier2), connection)
display(df_tier2)
print(f"Tier 2 assigned: {len(df_tier2)} users")
print(df_tier2['assigned_perk'].value_counts())
print(df_tier2['assignment_reason'].value_counts())

,user_id,assigned_perk,life_total_trips,assignment_reason
0,106907,No Change Fees,2,tier_2_two_trip_high_cancel
1,133058,Free Checked Bags,2,tier_2_two_trip_exclusive_intl
2,174997,No Change Fees,2,tier_2_two_trip_high_cancel
3,189676,Exclusive Discounts,2,tier_2_two_trip_default
4,204997,No Change Fees,2,tier_2_two_trip_high_cancel
...,...,...,...,...
1310,774666,Exclusive Discounts,2,tier_2_two_trip_default
1311,777846,Exclusive Discounts,2,tier_2_two_trip_default
1312,780167,Exclusive Discounts,2,tier_2_two_trip_default
1313,785186,Exclusive Discounts,2,tier_2_two_trip_default


Tier 2 assigned: 1315 users
assigned_perk
Exclusive Discounts    936
Free Checked Bags      302
No Change Fees          77
Name: count, dtype: int64
assignment_reason
tier_2_two_trip_default           936
tier_2_two_trip_exclusive_intl    302
tier_2_two_trip_high_cancel        77
Name: count, dtype: int64


#### Tier 3 - Behavioral perk assignment

##### Note about the Tier 3 shift  

The stats around this tier group are higher revenues, lower discount demands, higher loyalty and trips per year.   
Perk assignment prioritizes what each tier user most likely needs to continue or improve their completed travel history.  

1. First priority = increasing completed trips.  
    - 40%+ cancellations = No change fees  
2. Second priority = incentivize the unique segment of majority international travelers with Free checked bags.  
    - 90%+ completed international flights (open consideration to lower threshold to 80%)  
3. Third priority = support the unique segment that is primarily hotel booking with Free hotel meals.  
    - 50%+ bookings include hotel discounts but not primarily an International traveler and not discount dependant.  
4. Fourth priority = retain the primarily flight booking discount seeking travelers with continued Exclusive discounts.   
    - Currently this is a small subsect of non-international travelers that are mostly discount dependant.   
    - 50%+ bookings include flight discounts, but not a big hotel discount seeker and not a majority international traveler.  
5. Final priority = retain and incentivize the high loyalty, non-discount seeking group of travelers that remain by exclusion.  
    - not a majority international traveler, with less than 50% of any bookings discounted, and less than 40% cancellation rate  
    - the priority with this group is retention by adding value to their existing consistent travel patterns.  
    - Premium Travel Benefits to encourage the next booking with creature comforts and special benefits.  

In [19]:
q_tier3 = """
SELECT
    user_id,
    CASE
        WHEN COALESCE(cancel_spend_ratio, 0) >= 0.40 
            THEN 'No Change Fees'
        WHEN COALESCE(pct_complete_flights_intl, 0) >= 0.90 
            THEN 'Free Checked Bags'
        WHEN hotel_discount_rate >= 0.50 
            AND COALESCE(flight_discount_rate, 0) < 0.50
            AND COALESCE(pct_complete_flights_intl, 0) < 0.90
            THEN 'Free Hotel Meals'
        WHEN flight_discount_rate >= 0.50
            AND COALESCE(hotel_discount_rate, 0) < 0.50
            AND COALESCE(pct_complete_flights_intl, 0) < 0.90
            THEN 'Exclusive Discounts'
        ELSE 'Premium Travel Benefits'
    END AS assigned_perk,
    life_total_trips,
    CASE
        WHEN COALESCE(cancel_spend_ratio, 0) >= 0.40 
            THEN 'tier3_high_cancel_spend'
        WHEN COALESCE(pct_complete_flights_intl, 0) >= 0.90 
            THEN 'tier3_near_exclusive_intl'
        WHEN hotel_discount_rate >= 0.50 
            AND COALESCE(flight_discount_rate, 0) < 0.50
            AND COALESCE(pct_complete_flights_intl, 0) < 0.90
            THEN 'tier3_high_hotel_discount'
        WHEN flight_discount_rate >= 0.50
            AND COALESCE(hotel_discount_rate, 0) < 0.50
            AND COALESCE(pct_complete_flights_intl, 0) < 0.90
            THEN 'tier3_high_flight_discount'
        ELSE 'tier3_high_value_loyal'
    END AS assignment_reason
FROM tt_project.user_features
WHERE life_total_trips >= 3;
"""
df_tier3 = pd.read_sql(sa.text(q_tier3), connection)
display(df_tier3)
print(f"Tier 3+ assigned: {len(df_tier3)} users")
print(df_tier3['assigned_perk'].value_counts())
print(df_tier3['assignment_reason'].value_counts())

,user_id,assigned_perk,life_total_trips,assignment_reason
0,23557,Free Hotel Meals,3,tier3_high_hotel_discount
1,94883,Premium Travel Benefits,3,tier3_high_value_loyal
2,101486,Free Checked Bags,3,tier3_near_exclusive_intl
3,101961,Premium Travel Benefits,7,tier3_high_value_loyal
4,118043,Free Hotel Meals,5,tier3_high_hotel_discount
...,...,...,...,...
3217,760793,Premium Travel Benefits,3,tier3_high_value_loyal
3218,767426,Premium Travel Benefits,4,tier3_high_value_loyal
3219,785107,Premium Travel Benefits,4,tier3_high_value_loyal
3220,792549,Premium Travel Benefits,4,tier3_high_value_loyal


Tier 3+ assigned: 3222 users
assigned_perk
Premium Travel Benefits    2283
Free Checked Bags           413
Exclusive Discounts         249
Free Hotel Meals            221
No Change Fees               56
Name: count, dtype: int64
assignment_reason
tier3_high_value_loyal        2283
tier3_near_exclusive_intl      413
tier3_high_flight_discount     249
tier3_high_hotel_discount      221
tier3_high_cancel_spend         56
Name: count, dtype: int64


### Combine Perk Assignments

In [21]:
df_all_perks = pd.concat([df_tier0, df_tier1, df_tier2, df_tier3], ignore_index=True)

print(f"Total users assigned: {len(df_all_perks)}")
print(f"Expected: 5,998")
print(f"Difference: {5998 - len(df_all_perks)}")
print()
print(df_all_perks['assigned_perk'].value_counts())
print()
print(df_all_perks['assignment_reason'].value_counts())

Total users assigned: 5998
Expected: 5,998
Difference: 0

assigned_perk
Exclusive Discounts        2546
Premium Travel Benefits    2283
Free Checked Bags           715
No Change Fees              233
Free Hotel Meals            221
Name: count, dtype: int64

assignment_reason
tier3_high_value_loyal            2283
tier_2_two_trip_default            936
tier_1_single_trip_default         839
tier_0_never_booked                522
tier3_near_exclusive_intl          413
tier_2_two_trip_exclusive_intl     302
tier3_high_flight_discount         249
tier3_high_hotel_discount          221
tier_1_single_trip_high_cancel     100
tier_2_two_trip_high_cancel         77
tier3_high_cancel_spend             56
Name: count, dtype: int64


## Export

### Export perk assignments to table

In [ ]:
from sqlalchemy import text

df_all_perks.to_sql(
    name='perk_assignments',
    con=connection,
    schema='tt_project',
    if_exists='replace',
    index=False
)

# Verify
verify = pd.read_sql(
    sa.text("SELECT assigned_perk, COUNT(*) as users FROM tt_project.perk_assignments GROUP BY assigned_perk ORDER BY users DESC"),
    connection
)
display(verify)

print(f"Total rows in perk_assignments table: {verify['users'].sum()}")

,assigned_perk,users
0,Exclusive Discounts,2546
1,Premium Travel Benefits,2283
2,Free Checked Bags,715
3,No Change Fees,233
4,Free Hotel Meals,221


Total rows in perk_assignments table: 5998


### Export Perk Assignments table to csv

In [ ]:
q_export = """
SELECT 
    user_id,
    assigned_perk,
    life_total_trips,
    assignment_reason
FROM tt_project.perk_assignments
ORDER BY user_id;
"""
os.makedirs(DATA_PATH, exist_ok=True)
df_perk_export = pd.read_sql(sa.text(q_export), connection)
df_perk_export.to_csv(
    os.path.join(data_path, 'perk_assignments.csv'),
    index=False)
print(f"Exported: {len(df_perk_export)} rows")
print(df_perk_export['assigned_perk'].value_counts())

Exported: 5998 rows
assigned_perk
Exclusive Discounts        2546
Premium Travel Benefits    2283
Free Checked Bags           715
No Change Fees              233
Free Hotel Meals            221
Name: count, dtype: int64


# THE END